# WFP Syria — data exploration and model starter

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gromicho/teaching/blob/main/courses/aabw/notebooks/wfp-syria/starter-data-visualization.ipynb)

**Maintained by the Advanced Analytics for a Better World teaching team**  
AABW teaching notebook · Version 2026.1

Explores the WFP Syria network data and prepares the nominal optimization model.


## Download data and import the required packages


In [ ]:
%pip install -q pyomo highspy xlsxwriter

from pathlib import Path
from urllib.request import urlretrieve
import sys

at_colab = "google.colab" in sys.modules
figure_path = "."

# One canonical copy of every dataset and the shared workbook helper.
import hashlib
from urllib.request import urlopen
RESOURCE_FILES = {'Data Set Feedcalculator.xlsx': ('data/cases/feed-calculator.xlsx', '2e3653fe1b3a9995abedfb5902157dea2616c72550b62b7b060442e281bf8fe7'), 'DataSetFeedCalculator.xlsx': ('data/cases/feed-calculator.xlsx', '2e3653fe1b3a9995abedfb5902157dea2616c72550b62b7b060442e281bf8fe7'), 'DataErmeraTimorLeste.xlsx': ('data/cases/timor-leste.xlsx', '31431b07df5b6797630b9ec5023f81aacd76d5bdab4e21cdd3d9fe4e129f021a'), 'DataSyriaCaseWFP.xlsx': ('data/wfp-syria/DataSyriaCaseWFP.xlsx', 'f3e604eb3000d20e0e7eb64c7f847b6e34b83b85a417031a77805b8ea614c55b'), 'WFPCleanWithLocations.xlsx': ('data/wfp-syria/WFPCleanWithLocations.xlsx', 'aa46234010b4b4a94558dab49e0211792ac28bd8a34280dc0927aeca52c0bd8f'), 'WFP_Locations.xlsx': ('data/wfp-syria/WFP_Locations.xlsx', '029fc6ad26c992f0e460954aae9a5d6e1c6bca8a5728c814bd1f233ff54a3ceb'), 'util_AABW.py': ('support/util_AABW.py', '36bce9b87106d200ba5f8c34da6e3990550c6db8045b70041075cb4d1fcef820')}

def fetch_course_file(filename):
    relative, expected = RESOURCE_FILES[filename]
    path = Path(filename)
    if not path.is_file():
        local = Path(relative)
        if local.is_file():
            payload = local.read_bytes()
        else:
            url = 'https://raw.githubusercontent.com/gromicho/teaching/main/' + relative
            with urlopen(url, timeout=45) as response:
                payload = response.read()
        if hashlib.sha256(payload).hexdigest() != expected:
            raise ValueError('Unexpected teaching resource: ' + filename)
        path.write_bytes(payload)
    if hashlib.sha256(path.read_bytes()).hexdigest() != expected:
        raise ValueError('Unexpected local teaching resource: ' + filename)
    return path

for filename in ("util_AABW.py", "DataSyriaCaseWFP.xlsx", "WFP_Locations.xlsx"):
    fetch_course_file(filename)

import util_AABW as abw
import pandas as pd
import numpy as np
import difflib
import seaborn as sns
import matplotlib.pyplot as plt
import pprint
import pyomo.environ as pyo

pp = pprint.PrettyPrinter(indent=2, width=128, compact=True, sort_dicts=True)


# Prepare the data

In [ ]:
data_file = 'DataSyriaCaseWFP.xlsx'

data = abw.ReadWorkbookIntoNamedTuple( data_file )
coordinates = abw.ReadWorkbookIntoNamedTuple( 'WFP_Locations.xlsx' ).Sheet1

### COORDINATE DATA ####
#vary a little between suppliers, transshipmentnodes and demand nodes to make the differences clear when plotting
d_correction = {' D': 0.02,
                ' TS': -0.02}

for n, c in zip(d_correction.keys(), d_correction.values()):
  b = coordinates.NameType.str.contains(n)
  coordinates.loc[b, 'latitude'] = coordinates.loc[b, 'latitude'] + c
  coordinates.loc[b, 'longitude']= coordinates.loc[b, 'longitude']+ abs(c)

data._fields

Interpretations of NodesTypes Types:
* I = International supplier
* R = Regional supplier
* L = Local market (both supply and deliver)
* D = Delivery point
* TS = Transshipment node

## Clean the data

The issues we deal with with this data are:
- Node names do not always match:
  * some of the node names listed in columns A and B of the Edges - Cost sheet do not math the names on the Nodes - Types sheet
  * not all nodes are included in the markets
  * the names listed in columns of FoodCost are not the same as in NodeTypes
- Food names do not always match: discrepancies between the food names in the nutritional table and in the international price table.
  * Some food names in FoodNutritionalValue and FoodInternationalPrice differ
  * Nutrients in the NutrientRequirements that can not be found in FoodNutritionalValue
- Coordinates are missing from the NodesTypes dataset


## Showing the problems

#### Node and edge names differ sometimes

In [ ]:
set_nodes = set(data.NodesTypes.Name)
set_start_nodes = set(data.EdgesCost.A)
set_end_nodes = set(data.EdgesCost.B)
set_nodes_in_edges = set_start_nodes | set_end_nodes

pd.DataFrame.from_dict(
    { 'In nodes but not in edges' : sorted(set_nodes-set_nodes_in_edges),
      'In edges but not in nodes' : sorted(set_nodes_in_edges-set_nodes) },
    orient='columns'
)

#### Market names differ from node names

In [ ]:
set_markets = set(data.FoodCost.adm1_name)
set_node_names = set(' '.join(n.split(' ')[:-1]) for n in set_nodes)

pd.DataFrame.from_dict(
    { 'In nodes but not in markets' : dict(zip(range(1000),sorted(set_node_names-set_markets))),
      'In markets but not in nodes' : dict(zip(range(1000),sorted(set_markets-set_node_names))),
      'In markets and in nodes' : dict(zip(range(1000),sorted(set_markets&set_node_names))) },
    orient='columns'
).fillna('')

#### Differences FoodNutritionalValue names and FoodInternationalPrice names

In [ ]:
pd.DataFrame.from_dict(
    { 'In nutritional but not in price' :
        sorted(set(data.FoodNutritionalValue.Food) -
               set(data.FoodInternationalPrice.Food)),
      'In price but not in nutritional' :
        sorted(set(data.FoodInternationalPrice.Food) -
               set(data.FoodNutritionalValue.Food)) },
    orient='columns'
)

#### Nutrients in the NutrientRequirements that can not be found in FoodNutritionalValue


In [ ]:
nutrient_requirements = data.NutrientRequirements.T.drop(labels='Type',axis=0)[0].to_dict()

{ n : nutrient_requirements[n] for n in
  nutrient_requirements.keys() - set(data.FoodNutritionalValue.columns) }

## Clean the data

#### Universalize the names of the nodes

#### Universalize the names of the foods

## Set indexes

#### Add coordinates to the data

# Visualize the network

In [ ]:
# set the abbreviations, colors and legend names
abbreviation = {
    'Aleppo' : 'Al',
    'Amman' : 'Am',
    'Ar Raqqa' : 'AR',
    'As_Suweida' : 'AS',
    'Beirut' : 'Be',
    'Damascus' : 'Dm',
    'Daraa' : 'Da',
    'Dayr_Az_Zor' : 'DZ',
    'Gaziantep' : 'Gz',
    'Hama' : 'Hm',
    'Hassakeh' : 'Hs',
    'Homs' : 'Ho',
    'Idleb' : 'Id',
    'Jubb_al_Jarrah' : 'JJ',
    'Qamishli' : 'Qi'
}

category_colors = dict(
    I= 'pink',
    R= 'lightgreen',
    L= 'orange',
    D= 'red',
    TS= 'lightblue'
  )

node_type = dict(
    I= 'International supplier',
    R= 'Regional supplier',
    L= 'Local market (both supply and deliver)',
    D= 'Delivery point',
    TS= 'Transshipment node'
)

#### Using Folium

In [ ]:
if "coordinates" in data.NodesTypes.columns:
    import folium as fl
    import folium.plugins
    import pandas as pd

    def add_markers_from_dict(Map, df, d_col = category_colors):

      for n,c,t in zip(df.index, df.coordinates, df.Type) :
        fl.Marker(c,
                  icon=fl.plugins.BeautifyIcon(icon='',
                                               icon_shape='circle',
                                               background_color=d_col[t],
                                               border_width=0,
                                               inner_icon_style='font-size:15px; text-align:center'),
                  tooltip=n).add_to(Map)

      d = df.coordinates.to_dict()
      for a,b in data.EdgesCost.index:
        fl.PolyLine([d[a], d[b]], color="black", weight=2.5, opacity=1).add_to(Map)
      return Map

    def add_legend(d_col = category_colors, node_type = node_type):
      legend_html = """
      <div style="position: fixed; bottom: 160px; left: 80px; z-index:9999; font-size: 22px;">
      <p><strong>Legend</strong></p>
      """
      for category, color in d_col.items():
          legend_html += f'<i class="fa fa-circle fa-1x" style="color:{color}"></i> {node_type[category]}<br>'
      legend_html += "</div>"
      return legend_html

    # c = geopy.Nominatim(user_agent='user_agent').geocode('Syria')
    c = (coordinates.latitude.mean(), coordinates.longitude.mean())
    Map = fl.Map(location=c, zoom_start=7)
    Map = add_markers_from_dict(Map, data.NodesTypes)
    legend_html = add_legend()
    Map.get_root().html.add_child(folium.Element(legend_html))
    Map
else:
    print("Visualization skipped: complete the data-preparation steps first.")


#### Using Networkx

In [ ]:
if "coordinates" in data.NodesTypes.columns:
    import networkx as nx
    import matplotlib.patches as mpatches

    def create_graph(data):
      # Create graph
      g = nx.DiGraph()

      for node, row in data.NodesTypes.iterrows():
          g.add_node(node,**row.to_dict())
      for edge, row in data.EdgesCost.iterrows():
          g.add_edge(*edge,**row.to_dict())
      return g

    g = create_graph( data)

    #create essential functions and names
    pure_name = lambda n : ' '.join( n.split(' ')[:-1] )
    type_from_name = lambda n : n.split(' ')[-1]

    #create information for the legend
    node_labels = { n : abbreviation[pure_name(n)] for n in data.NodesTypes.index }
    node_colors = [ category_colors[t] for t in data.NodesTypes.Type ]

    #create positions for the nodes
    pos = dict()
    for name, (lat, lon) in zip(data.NodesTypes.index, data.NodesTypes.coordinates):
        pos[name] = [lon,lat]

    pos = nx.spring_layout(g, pos=pos,
                           k=13, scale=8, weight='tCost',
                           iterations=2, seed=2023)

    #plot the figure
    plt.figure(figsize=(10,6))

    nx.draw(g, pos, with_labels=False, node_size=500, node_color=node_colors)
    nx.draw_networkx_labels(g, pos, labels=node_labels)
    nx.draw_networkx_edges(g, pos,
    width=[1e-5+5e-3*c for c in nx.get_edge_attributes(g,'tCost').values()],
    edge_color='black'
    )

    legend_elements = [ mpatches.Patch(color=color, label=node_type[category])
        for category, color in category_colors.items() ]

    # Add the legend to the plot
    plt.legend(handles=legend_elements, loc='lower right')

    # Export explicitly with plt.savefig(...) when a figure file is needed.

    plt.show()

else:
    print("Visualization skipped: complete the data-preparation steps first.")


# Implement the nominal model

## Model:

\begin{array}{l l l l }
\min_{F,R} &    \sum_{i \in N_S, j, k} pc_{ik}F_{ijk} + \sum_{i,j,k} tc_{ijk} F_{ijk} + \sum_{i,j,k} hc_j F_{ijk}              &   &\qquad \text{minimize costs} \\
 s.t. & \sum_{i\in N: ij\in E}\sum_{k\in K} F_{ijk} = \sum_{i\in N: ji\in E} \sum_{k\in K}F_{jik} & \forall j \in N_{T} &\qquad  \text{Flow is preserved in transshipment nodes}\\
& \sum_{i\in N: ij\in E} F_{ijk} \geq dem_j R_k & \forall j \in N_B, \forall k\in K &\qquad  \text{All beneficiaries receive food}\\
& \sum_{k\in K} nutval_{kl} R_k \geq nutreq_l & \forall l \in L &\qquad \text{Nutritional requirements}\\
& R_k, F_{ijk}\geq 0 &\forall i,j\in N, \forall k\in K&\qquad  \text{Flow and rations are non-negative}\\
\end{array}


## Sets:
$$
\begin{align*}
    N &: \text{set of nodes.}&&&&&\\
    N_S &: \text{set of suppliers.}&&&&&\\
    N_T &: \text{set of transshipment points.}&&&&&\\
    N_B &: \text{set of beneficiary camps.}&&&&&\\
    E &: \text{set of edges.}&&&&&\\
    L &: \text{set of nutrients.}&&&&&\\
    K &: \text{set of commodities.}&&&&&
\end{align*}
$$

## Parameters:
$$
\begin{array}{l l l l}
    dem_i &: \text{number of beneficiaries at node } i \in N_B. &\\
    hc_i &: \text{costs of handling at node $i \in N$ \ $ N_S$ (dollars/kg).} & \\
    pc_{ik} &: \text{costs of procuring commodity $k\in K$ from node $i\in N_S$ (dollars kg).} & \\
    tc_{ijk} &: \text{costs of transporting commodity $k\in K$ from node $i\in N_S$ to node $j\in N$ (dollars kg).} & \\
    nutreq_l &: \text{nutritional requirements of beneficiary for nutrient $l\in L$.} & \\
    nutval_{kl} &: \text{nutritional value of commodity $k\in K$ for nutrient $l\in L$ (per kg).} &
\end{array}
$$

## Variables:
$$
\begin{align*}
    F_{ijk} &: \text{amount of commodity $k\in K$ sent from node $i\in N$ to $j\in N$ (kg).}&\\
    R_k &: \text{Ration size of commodity $k\in K$ (kg).}&
\end{align*}
$$

In [ ]:
def WFP_model(data):
    """Build and return the nominal WFP model."""
    # TODO: implement the sets, parameters, variables, objective, and constraints.
    raise NotImplementedError("Complete the nominal WFP model before solving it.")


In [ ]:
m = WFP_model( data )

In [ ]:
solver = pyo.SolverFactory("appsi_highs")
results = solver.solve(m)
pyo.assert_optimal_termination(results)


In [ ]:
fin_F = {(i,j,k): pyo.value(m.F[i,j,k]) for i,j in m.E for k in m.K if  pyo.value(m.F[i,j,k])>0}
fin_R = {k: pyo.value(m.R[k]) for k in m.K if pyo.value(m.R[k])>0}

In [ ]:
print('Costs of designing the supply chain like this: $' + str(round(pyo.value(m.Cost),2)))

### Visualize the results
can plot all different commodity supply chains together (but the colors overlap), or dan show them separately. To plot them all, pass along the argument 'all', otherwise choose one present in `fin_R.keys()`.

In [ ]:
import folium as fl
import pandas as pd


# The Robust Model:
<!--
\begin{array}{l l l l }
\min_{F,R} &    q + \sum_{i \in N_{SLR}, j, k} \mu_{ik1}F_{ijk1} + \sum_{i \in N_{SI}, j, k} pc_{ik}F_{ijk} + \sum_{i,j,k} tc_{ijk} F_{ijk} + \sum_{i,j,k} hc_j F_{ijk}              &   &\qquad \text{minimize costs} \\
 s.t. & \mu^T F^P + \Omega \mid\mid chol(\Sigma) F^P\mid\mid_2  \leq q  &  &\qquad  \text{Robust counterpart}\\
 & \sum_i F_{ijk} = \sum_i F_{jik} & \forall j \in N_{TS}, \forall k\in K &\qquad  \text{Flow is preserved}\\
& \sum_i F_{ijk} \geq dem_j R_k & \forall j \in N_B, \forall k\in K &\qquad  \text{All beneficiaries receive food}\\
& \sum_k nutval_{kl} R_k \geq nutreq_l & \forall l \in L &\qquad \text{Nutritional requirements}\\
& R_k, F_{ijk}\geq 0 &\forall i,j\in N, \forall k\in K&\qquad  \text{Flow and rations are non-negative}\\
\end{array}


## Sets:
$$
\begin{align*}
    N &: \text{set of nodes.}&\\
    N_{SI} &: \text{set of international suppliers.}&\\
    N_{SLR} &: \text{set of local and regional suppliers.}&\\
    N_T &: \text{set of transshipment points.}&\\
    N_B &: \text{set of beneficiary camps.}& \\
    N_{SB} &: \text{set of suppliers and beneficiary camps.}& \\
    L &: \text{set of nutrients.}& \\
    K &: \text{set of commodities.}& \\
    T &: \text{set of time periods in which can be bought.}& \\
    U &= \{\zeta \mid \zeta^{T}\Sigma^{-1} \zeta \leq \Omega^2 \}, \text{uncertainty set} &
\end{align*}
$$

## Parameters:
$$
\begin{array}{l l l l}
    dem_i &: \text{number of beneficiaries at node } i \in N_B. &\\
    hc_i &: \text{costs of handling at node $i \in N$ \ $ N_S$ (dollars/kg).} & \\
    tc_{ijk} &: \text{costs of transporting commodity $k\in K$ from node $i\in N_S$ to node $j\in N$ (dollars kg).} & \\
    nutreq_l &: \text{nutritional requirements of beneficiary for nutrient $l\in L$.} & \\
    nutval_{kl} &: \text{nutritional value of commodity $k\in K$ for nutrient $l\in L$ (per kg).} & \\
    pc_{ik} &: \text{costs of procuring commodity $k\in K$ from international supplier node $i\in N_SI$ (dollars kg).} & \\
    \mu_{ikt} &: \text{costs of procuring commodity $k\in K$ from local or regional supplier node $i\in N_SLR$ at time $t\in T$ (dollars kg).} &\\
    \Sigma &: \text{covariance matrix of the prices} &
\end{array}
$$

## Variables:
$$
\begin{align*}
    F_{ijkt} &: \text{amount of commodity $k\in K$ sent from node $i\in N$ to $j\in N$ at time $t\in T$ (kg).}&\\
    q &: \text{Robust variable that quantifies the price } &\\
    R_k &: \text{Ration size of commodity $k\in K$ (kg).}&
\end{align*}
$$ -->




In [ ]:
def robust_WFP_model(data, rho):
    """Build and return the robust WFP model for radius ``rho``."""
    # TODO: extend the nominal formulation with the robust counterpart.
    raise NotImplementedError("Complete the robust WFP model before solving it.")


In [ ]:
m_rob = robust_WFP_model( data , 2)

In [ ]:
# The ellipsoidal robust counterpart is a conic/quadratic model.
# Install Gurobi and make a valid academic or Web License Service license available
# in the runtime before executing this cell.
%pip install -q gurobipy
robust_solver = pyo.SolverFactory("gurobi_direct")
results = robust_solver.solve(m_rob)
pyo.assert_optimal_termination(results)


In [ ]:
rob_F = {(i,j,k): pyo.value(m_rob.F[i,j,k]) for i,j in m_rob.E for k in m_rob.K if  pyo.value(m_rob.F[i,j,k])>0}
rob_R = {k: pyo.value(m_rob.R[k]) for k in m_rob.K if pyo.value(m_rob.R[k])>0}

### Analyze the results for different values of $\rho$

In [ ]:
rho_values = [0, 0.5, 1, 1.5, 2]
# TODO: after implementing robust_WFP_model, solve the model for every value
# in rho_values and collect total food, number of ingredients, and total cost.
rho_values


### Visualize the results

### See how your constraints turned out